In [2]:
from dotenv import load_dotenv
from openai import OpenAI


In [3]:
load_dotenv()

True

In [4]:
import os

In [5]:
google_api_key = os.getenv("GOOGLE_API_KEY")

In [6]:
client = OpenAI(api_key = google_api_key,base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

messages = [{"role":"system","content":"You are a helpful assistant and Your name is Emma"},{"role":"user","content":"Hello,how are you!"}]

responses = client.chat.completions.create(
    model = "gemini-2.5-flash",
    messages = messages
)

In [7]:
result = responses.choices[0].message.content
print(result)

Hello! I'm doing great, thank you for asking!

My name is Emma, it's nice to meet you. How are you doing today?


<h3>Creating a simple agent by using simple classes</h3>

In [8]:
import re
import httpx

In [9]:
class Agent:
    def __init__(self,system=''):
        self.system =system
        self.messages = []

        if self.system:
            self.messages.append({'role':'system','content':system})


    def __call__(self,prompt):
        self.messages.append({'role':'user','content':prompt})
        result = self.execute()
        self.messages.append({'role':'assistant','content':result})
        return result

    def execute(self,model="gemini-2.5-flash",temperature=0):
        response = client.chat.completions.create(
            model =model,
            temperature=temperature,
            messages = self.messages
        )

        return response.choices[0].message.content

In [25]:
prompt = '''You are an helpful assitant ,and your name is Emma
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_cost:
e.g. get_cost: book
returns the cost of a book

wikipedia:
e.g. wikipedia: LangChain
Returns a summary from searching Wikipedia

Always look things up on Wikipedia if you have the opportunity to do so.

Example session #1:

Question: How much does a pen cost?
Thought: I should look the pen cost using get_cost
Action: get_cost: pen
PAUSE

You will be called again with this:

Observation: A pen costs $5

You then output:

Answer: A pen costs $5


Example session #2

Question: What is the capital of France?
Thought: I should look up France on Wikipedia
Action: wikipedia: France
PAUSE

You will be called again with this:

Observation: France is a country. The capital is Paris.

You then output:

Answer: The capital of France is Paris
'''.strip()


In [11]:
def wikipedia(q):
    response = httpx.get('https://en.wikipedia.org/w/api.php',params={
        'action':'query',
        'list':'search',
        'srsearch':q,
        'format':'json'
    })
    results=response.json().get('query').get('search',[])

    if not results:
        return None;
    return results[0]['snippet']

In [12]:
wikipedia('Statue of liberty')

'The <span class="searchmatch">Statue</span> <span class="searchmatch">of</span> <span class="searchmatch">Liberty</span> (<span class="searchmatch">Liberty</span> Enlightening the World; French: La Liberté éclairant le monde) is a colossal neoclassical sculpture on <span class="searchmatch">Liberty</span> Island in'

In [13]:
known_actions={
    'wikipedia':wikipedia
}

In [26]:
myagent = Agent(prompt)

In [27]:
myagent('What is your name?')

'Answer: My name is Emma.'

In [28]:
myagent.messages

[{'role': 'system',
  'content': 'You are an helpful assitant ,and your name is Emma\nYou run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_cost:\ne.g. get_cost: book\nreturns the cost of a book\n\nwikipedia:\ne.g. wikipedia: LangChain\nReturns a summary from searching Wikipedia\n\nAlways look things up on Wikipedia if you have the opportunity to do so.\n\nExample session #1:\n\nQuestion: How much does a pen cost?\nThought: I should look the pen cost using get_cost\nAction: get_cost: pen\nPAUSE\n\nYou will be called again with this:\n\nObserva

In [29]:
abot = Agent(prompt)

In [30]:
query = " What is the capital of France?"
abot(query)

'Action: wikipedia: France\nPAUSE'

In [31]:
next_prompt = f'Observation :{wikipedia(query)}'
print(next_prompt)

Observation :variants <span class="searchmatch">of</span> <span class="searchmatch">the</span> above closed-ended questions that possess specific responses are: On <span class="searchmatch">what</span> day were you born? (&quot;Saturday.&quot;) <span class="searchmatch">What</span> <span class="searchmatch">is</span> <span class="searchmatch">the</span> <span class="searchmatch">capital</span> <span class="searchmatch">of</span> <span class="searchmatch">France</span>? (&quot;Paris


In [32]:
abot(next_prompt)

'Answer: The capital of France is Paris'

In [33]:
abot.messages

[{'role': 'system',
  'content': 'You are an helpful assitant ,and your name is Emma\nYou run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_cost:\ne.g. get_cost: book\nreturns the cost of a book\n\nwikipedia:\ne.g. wikipedia: LangChain\nReturns a summary from searching Wikipedia\n\nAlways look things up on Wikipedia if you have the opportunity to do so.\n\nExample session #1:\n\nQuestion: How much does a pen cost?\nThought: I should look the pen cost using get_cost\nAction: get_cost: pen\nPAUSE\n\nYou will be called again with this:\n\nObserva

<h4>Automating the Agent</h4>

In [35]:
#defining regex to select Action:
action_re = re.compile(r'^Action: (\w+): (.*)$')

In [36]:
def query(question,max_turns=5):
    i =0
    bot = Agent(prompt)
    next_prompt = question

    while i<max_turns:
        i +=1
        result = bot(next_prompt)
        print(result)

        actions = [
            action_re.match(a) for a in result.split('\n') if action_re.match(a)
        ]

        if actions:
            action,action_input = actions[0].groups()

            if action not in known_actions:
                raise Exception(f'Unknown action: {action}: {action_input}')
            print(f' --running (action) {action_input}')
            observation = known_actions [action] (action_input)
            print(f'Observation: (observation)')
            next_prompt = f'Observation: {observation}'

        else:
            return

In [37]:
question = '''Hi,I wanted to know about about Statue of Liberty?''' 
query(question)

Action: wikipedia: Statue of Liberty
PAUSE
 --running (action) Statue of Liberty
Observation: (observation)
Answer: The Statue of Liberty (Liberty Enlightening the World; French: La Liberté éclairant le monde) is a colossal neoclassical sculpture on Liberty Island.
